# 検出器データ解析ノートブック (ACQ 版) — `.dat` ファイル

このノートブックは、Red Pitaya で取得した **`.dat` ファイル (CSV)** を解析するためのものです。`detector-acquisition-dma.ipynb` の **Option A (標準 ACQ)** で保存されたデータを対象にしています。

## 何が入っているデータ？

`.dat` ファイルには、1 ショット (1 回のトリガ取得) ごとに次の値が **1 行ずつ** 並んでいます。

| 列名 | 意味 |
|---|---|
| `totaltime` | 計測開始からの経過時間 [秒] |
| `deadtime` | この時点までに「データを書き出している間に取りこぼした時間」の累計 [秒] |
| `sum_ch1` | ch1 の波形を全サンプル足し合わせた値 (≈ 信号の面積) |
| `max_ch1` | ch1 の波形のピーク値 [V] |
| `sum_ch2` | ch2 の波形を全サンプル足し合わせた値 |
| `max_ch2` | ch2 の波形のピーク値 [V] |

同じファイル名で `.json` (取得条件のメモ) も一緒に保存されています。

## このノートブックで体験すること

1. **データを読み込む** — pandas でテーブルとして開く。
2. **ヒストグラムを描く** — サム値・ピーク値の **分布** を見る。
3. **時間による変動を見る** — 取得レートやデッドタイム比を調べる。
4. **2 つのチャンネルの相関を取る** — ch1 と ch2 の関係を散布図で見る。
5. **しきい値カットを試す** — ノイズと信号を分ける体験。

> **読み方のコツ**: コードセルは上から順番に実行してください。マークダウンの説明 → コード → 結果、というリズムで進みます。


## 0. ライブラリの準備

このノートブックでは 3 つのライブラリを使います。

- **numpy** — 数値計算 (配列の足し算、平均など)
- **pandas** — 表形式データの便利な操作 (`.dat` を表として開く)
- **plotly** — グラフ描画 (マウスでズームできるインタラクティブな図)

もし `plotly` が入っていなかったら、下のセルの先頭の `#` を外して 1 度だけ実行してください。


In [ ]:
# import sys, subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "numpy", "pandas", "plotly"])


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("numpy", np.__version__, "/ pandas", pd.__version__)


## 1. データの読み込み

### 1.1 `./data/` の中で **一番新しい** `.dat` ファイルを自動的に探す

毎回ファイル名をタイプするのは面倒なので、`./data/` フォルダの中から **更新時刻が最新** の `.dat` を自動で見つける関数を作ります。
特定のファイルを開きたいときは、自動検出を使わず `dat_path = Path('./data/2026-04-28-140000.dat')` のように直接指定してもかまいません。


In [ ]:
def find_latest(ext: str, folder: str = "./data") -> Path:
    '''folder 内で拡張子 ext を持つファイルのうち、更新時刻が最新のものを返す。'''
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(
            f"{folder} が見つかりません。取得用ノートブックでデータを取った後に実行してください。"
        )
    candidates = sorted(folder.glob(f"*{ext}"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"{folder} の中に *{ext} ファイルが見つかりません。")
    return candidates[0]

dat_path = find_latest(".dat")
print("読み込むファイル:", dat_path)


### 1.2 メタ情報 (取得条件) を表示

同じ名前の `.json` があれば、取得開始時刻やトリガ条件などを表示します (なくても動きます)。


In [ ]:
json_path = dat_path.with_suffix(".json")
if json_path.exists():
    meta = json.loads(json_path.read_text())
    for key in ("start_datetime", "trigger_src", "trigger_level_V", "duty",
                "input_range_jumper", "decimation_macro"):
        if key in meta:
            print(f"  {key:24s}: {meta[key]}")
else:
    meta = {}
    print("(対応する .json が見つかりませんでした。データの読み込みは続行します。)")


### 1.3 表として読み込む

CSV 形式なので、pandas の `read_csv` で **表 (DataFrame)** として開けます。
最初の数行と、各列の **平均・最大・最小** などの基本統計量も確認しましょう。


In [ ]:
df = pd.read_csv(dat_path)
print("行数 (ショット数):", len(df))
print("列名:", list(df.columns))
df.head()


In [ ]:
df.describe()


## 2. ヒストグラム — サム値・ピーク値の分布

ヒストグラムは「どの値が何回ぐらい出たか」を棒グラフで表したもので、検出器のデータ解析の出発点です。

- **サム値 (`sum_ch1`, `sum_ch2`)**: 波形の面積 ≈ 信号のエネルギーに対応します。
- **ピーク値 (`max_ch1`, `max_ch2`)**: 信号の最大電圧。

ビン (棒の数) は **データ数の平方根** を目安にしました (`int(sqrt(N))`)。これは Sturges' rule などより素朴ですが、高校生にも分かりやすい目安です。


In [ ]:
nbins = max(20, int(np.sqrt(len(df))))
fig = make_subplots(rows=2, cols=2, subplot_titles=(
    "sum_ch1 (ch1 のサム)", "sum_ch2 (ch2 のサム)",
    "max_ch1 (ch1 のピーク [V])", "max_ch2 (ch2 のピーク [V])"))

fig.add_trace(go.Histogram(x=df["sum_ch1"], nbinsx=nbins, name="sum_ch1"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["sum_ch2"], nbinsx=nbins, name="sum_ch2"), row=1, col=2)
fig.add_trace(go.Histogram(x=df["max_ch1"], nbinsx=nbins, name="max_ch1"), row=2, col=1)
fig.add_trace(go.Histogram(x=df["max_ch2"], nbinsx=nbins, name="max_ch2"), row=2, col=2)
fig.update_layout(width=1000, height=700, showlegend=False, title_text="サム値・ピーク値のヒストグラム")
fig.show()


**読み取りのヒント**

- 0 付近の鋭いピークは **ノイズ** や **空 (からっぽ) のショット** を表していることが多いです。
- それより右に裾を引く分布は **本物の信号** であることが期待されます。
- ch1 と ch2 で形が似ているなら、両方の検出器が同じ現象を見ている可能性が高いです。


## 3. 時間による変動

データを取っている **間** に何かおかしなことが起きていないか (例: 信号がだんだん弱くなる、急に増える) を、横軸=時間でプロットして確認します。
ここでは ch1 のピーク値を時間に対してプロットします。

加えて、計測の **平均レート (1 秒間に何ショット取れたか)** と **デッドタイム比** も表示します。


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scattergl(x=df["totaltime"], y=df["max_ch1"], mode="markers",
                           marker=dict(size=4, opacity=0.6), name="max_ch1"))
fig.update_layout(xaxis_title="totaltime [s]", yaxis_title="max_ch1 [V]",
                  width=1000, height=400, title_text="ピーク値の時間変動 (ch1)")
fig.show()


In [ ]:
total_seconds = float(df["totaltime"].iloc[-1])
total_dead    = float(df["deadtime"].iloc[-1])
n_shots       = len(df)

print(f"ショット数        : {n_shots}")
print(f"計測時間          : {total_seconds:.3f} s")
print(f"平均レート        : {n_shots / total_seconds:.2f} shots/s")
print(f"累計デッドタイム  : {total_dead:.3f} s  ({100*total_dead/total_seconds:.1f}%)")


## 4. 2 チャンネル間の相関

ch1 と ch2 を **同時に** 見ているとき、同じ粒子が両方のチャンネルにヒットしていれば、**サム値どうし・ピーク値どうしに正の相関** が出るはずです。
散布図を描いてその関係を確かめましょう。

### 相関係数

`numpy.corrcoef` で **相関係数** という数値を計算します。

- **+1 に近い** → 一方が増えると他方も増える (強い正の相関)
- **0 に近い** → 関係なし
- **-1 に近い** → 一方が増えると他方は減る (強い負の相関)


In [ ]:
r_sum = float(np.corrcoef(df["sum_ch1"], df["sum_ch2"])[0, 1])
r_max = float(np.corrcoef(df["max_ch1"], df["max_ch2"])[0, 1])

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"sum: ch1 vs ch2  (相関係数 r = {r_sum:.3f})",
    f"max: ch1 vs ch2  (相関係数 r = {r_max:.3f})"))
fig.add_trace(go.Scattergl(x=df["sum_ch1"], y=df["sum_ch2"], mode="markers",
                           marker=dict(size=4, opacity=0.5), name="sum"), row=1, col=1)
fig.add_trace(go.Scattergl(x=df["max_ch1"], y=df["max_ch2"], mode="markers",
                           marker=dict(size=4, opacity=0.5), name="max"), row=1, col=2)
fig.update_xaxes(title_text="ch1", row=1, col=1); fig.update_yaxes(title_text="ch2", row=1, col=1)
fig.update_xaxes(title_text="ch1", row=1, col=2); fig.update_yaxes(title_text="ch2", row=1, col=2)
fig.update_layout(width=1100, height=450, showlegend=False)
fig.show()

print(f"相関係数 (sum) = {r_sum:.4f}")
print(f"相関係数 (max) = {r_max:.4f}")


**考えてみよう**

- 相関係数が **0 に近い** ときは、ch1 と ch2 が関係のない 2 つの現象を見ている可能性があります。
- 散布図に **直線状** に並んでいる点は、ch1 と ch2 が常に同じ比率で反応しているサイン (例: 同じ粒子が両方を貫通)。
- **線から外れた点** は、片方だけにヒットした「片チャンネル事象」かもしれません。


## 5. しきい値カットを試す — ノイズと信号を分ける

ノイズに埋もれた信号を見つけ出すには、**ある一定の値より大きいピーク** だけを選び出すのが基本テクニックです。
これを **しきい値カット (threshold cut)** と呼びます。

下のセルでは `threshold` 変数の値を変えながら何度も実行してみて、ヒストグラムの形がどう変わるかを観察しましょう。


In [ ]:
threshold = 0.10  # [V] — 自由に変えてみよう

mask_pass = df["max_ch1"] > threshold
n_pass = int(mask_pass.sum())
n_total = len(df)

print(f"しきい値 max_ch1 > {threshold:.3f} V を通過: {n_pass} / {n_total}  ({100*n_pass/n_total:.1f}%)")

nbins2 = max(20, int(np.sqrt(n_total)))
fig = go.Figure()
fig.add_trace(go.Histogram(x=df["max_ch1"], nbinsx=nbins2, name="全データ", opacity=0.6))
fig.add_trace(go.Histogram(x=df.loc[mask_pass, "max_ch1"], nbinsx=nbins2,
                           name=f"max_ch1 > {threshold:.3f} V", opacity=0.6))
fig.update_layout(barmode="overlay", xaxis_title="max_ch1 [V]", yaxis_title="件数",
                  width=900, height=420, title_text="しきい値カットの効果")
fig.show()


## おさらいと次の一歩

ここまでで、

- データの **形** (どんな列があってどんな範囲か) を把握し、
- **ヒストグラム** で全体の分布を見て、
- **時間変動** で計測中の安定性を確かめ、
- **2 ch の相関** で物理的なつながりを推定し、
- **しきい値カット** でノイズと信号を切り分けました。

ここから先のチャレンジ:

- `sum_ch1` についてもしきい値カットを試して、`max_ch1` カットと結果を比べてみよう。
- ch2 にもしきい値カットを掛けて、両方を通過するイベントだけを残してみよう (**コインシデンス**)。
- 時系列のヒストグラムを **時間ごとに区切って** 描き直し、レートが安定しているかを目視で確認しよう。

## トラブルシューティング

### `./data/ が見つかりません` と出る
取得用ノートブック (`detector-acquisition-dma.ipynb`) の **Option A** を一度実行して、`.dat` ファイルを作ってからこのノートブックを実行してください。
別の場所のファイルを使いたい場合は、`find_latest` を使わず `dat_path = Path('...')` で直接指定すれば OK です。

### グラフが表示されない
JupyterLab の場合は次のセルを 1 度だけ実行してみてください:
```python
import plotly.io as pio
pio.renderers.default = "jupyterlab"   # クラシック Notebook なら "notebook_connected"
```
HTML として保存してブラウザで開く方法も確実です: `fig.write_html("out.html")`。

### `pip install` で時間がかかる
`numpy` の再ビルドが走ると数分かかることがあります。Red Pitaya 上で実行する場合は `--no-deps` オプション付きで Plotly を入れる方が早いです (取得用ノートブックの 0 章を参照)。
